In [0]:
%sql
-- 1. Crear la base de datos para la capa Gold de Salud
CREATE DATABASE IF NOT EXISTS health_fhir_gold;

-- 2. Tabla Gold 1: Resumen de admisiones y diagnósticos por tipo de encuentro
CREATE OR REPLACE TABLE health_fhir_gold.kpi_encuentros_por_tipo AS
SELECT 
    encounter_class,
    diagnosis_code,
    diagnosis_display,
    COUNT(DISTINCT patient_id) AS total_pacientes_unicos,
    COUNT(encounter_id) AS total_encuentros,
    current_timestamp() AS _calculated_at
FROM health_fhir_silver.clean_patients_encounters
GROUP BY encounter_class, diagnosis_code, diagnosis_display;

-- 3. Tabla Gold 2: Detalle analítico de duración de estancia médica por paciente
CREATE OR REPLACE TABLE health_fhir_gold.analytics_duracion_estancia AS
SELECT 
    patient_id,
    CONCAT(family_name, ', ', given_name) AS nombre_completo_paciente,
    gender,
    encounter_id,
    encounter_class,
    diagnosis_display,
    encounter_start,
    encounter_end,
    -- Calcular la duración de la estancia en horas
    ROUND((CAST(encounter_end AS LONG) - CAST(encounter_start AS LONG)) / 3600.0, 2) AS duracion_horas_estancia
FROM health_fhir_silver.clean_patients_encounters;

In [0]:
%sql
-- Ver los KPIs agregados por tipo de encuentro y diagnóstico
SELECT * FROM health_fhir_gold.kpi_encuentros_por_tipo;

-- Ver el detalle analítico de duración de estancias hospitalarias
SELECT * FROM health_fhir_gold.analytics_duracion_estancia;